<a href="https://colab.research.google.com/github/mcramireza1/Grupo7_Metodos2/blob/main/T4/Punto_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Solitones

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from pathlib import Path
from textwrap import dedent

L = 40.0
N = 256
delta = 0.2
T = 15.0

C = 0.02

x = np.linspace(0, L, N, endpoint=False)
dx = L / N
dt = C * dx*3 / (delta*2 + 1e-12)
nt = int(np.ceil(T / dt))
dt = T/ max(nt,1)

phi0 = np.cos(2*np.pi*x/L)
phi0 -= phi0.mean()

phi = phi0.copy()

def Dx(a):
  return (np.roll(a,-1) - np.roll(a,1))/(2*dx)
def Dxx(a):
  return (np.roll(a,-1) - 2*a + np.roll(a,1)) / (dx*dx)
def Dxxx(a):
  return Dx(Dxx(a))

def rhs(a):
  # u_t = - (u u_x + δ^2 u_xxx)
  return - (a * Dx(a) + (delta**2) * Dxxx(a))

def trapz_periodic(f):
  return f.mean()*L

def constantes(a):
  phix = Dx(a)
  masa = trapz_periodic(a)
  momento = trapz_periodic(a*a)
  energia = trapz_periodic((1/3)*a**3 - (delta*phix)*2)
  return masa, momento, energia

snapshots = []
ctes = []
t = 0.0
ctes.append((t,*constantes(phi)))
snap_stride = max(1, nt//80)

for n in range(1, nt+1):
  k1 = rhs(phi)
  k2 = rhs(phi + 0.5*dt*k1)
  k3 = rhs(phi + 0.5*dt*k2)
  k4 = rhs(phi + dt*k3)
  phi = phi + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
  t = n*dt
  if n % snap_stride == 0 or n == nt:
      snapshots.append((t, phi.copy()))
  ctes.append((T, *constantes(phi)))

out_dir =Path(".")
nombre = "3_solitones_kdv_fd"

png_path = out_dir / f"{nombre}.png"
pdf_path = out_dir / f"{nombre}.pdf"
inv_png_path = out_dir / f"{nombre}_invariantes.png"
mp4_path = out_dir / f"{nombre}.mp4"
gif_path = out_dir / f"{nombre}.gif"

plt.figure()
plt.plot(x, phi0, label="t=0")
plt.plot(x, phi, label=f"t={t:.2f}")
plt.xlabel("x"); plt.ylabel("u(t,x)")
plt.title("KdV (FD, periódicas)")

plt.legend(); plt.tight_layout()
plt.savefig(png_path, dpi=160)
plt.savefig(pdf_path)
plt.close()

inv_arr = np.array(ctes)
t_arr = inv_arr[:,0]
masa_arr = inv_arr[:,1]
momento_arr = inv_arr[:,2]
energia_arr = inv_arr[:,3]
plt.figure()
plt.plot(t_arr, masa_arr, label="masa ∫u dx")
plt.plot(t_arr, momento_arr, label="momento ∫u² dx")
plt.plot(t_arr, energia_arr,   label="energía ∫(1/3 u³ - (δ u_x)²) dx")
plt.xlabel("tiempo"); plt.title("Invariantes discretos")
plt.legend(); plt.tight_layout()
plt.savefig(inv_png_path, dpi=160)
plt.close()

fig, ax = plt.subplots()
line, = ax.plot([], [])
ax.set_xlim(0, L)
ax.set_ylim(np.min(phi0) - 0.5, np.max(phi0) + 0.5)
ax.set_xlabel("x"); ax.set_ylabel("phi")

def init():
    line.set_data([], [])
    return (line,)

def animate(i):
    t_i, phi_i = snapshots[i]
    line.set_data(x, phi_i)
    ax.set_title(f"Evolución de KdV (t={t_i:.2f})")
    return (line,)

anim = animation.FuncAnimation(
    fig, animate, init_func=init,
    frames=len(snapshots), interval=60, blit=True
)

saved_as = None
try:
    anim.save(mp4_path, writer="ffmpeg", dpi=160)
    saved_as = "mp4"
except Exception:
    try:
        anim.save(gif_path, writer="pillow", dpi=120)
        saved_as = "gif"
    except Exception:
        saved_as = "none"
plt.close(fig)

note = dedent(f"""
    Simulación KdV (diferencias finitas, condiciones periódicas)
    PDE: u_t + u u_x + δ^2 u_xxx = 0, δ = {delta}
    Dominio: x in [0,{L}), N = {N}, dx = {dx:.5f}
    Tiempo: RK4 explícito, dt = {dt:.3e}, pasos = {nt}
    Archivos: {png_path.name}, {pdf_path.name}, {inv_png_path.name},
              {'{'+mp4_path.name+'}' if saved_as=='mp4' else '{'+gif_path.name+'}' if saved_as=='gif' else '(sin animación)'}
    Invariantes (inicio → fin):
      masa     : {masa_arr[0]: .6e} → {masa_arr[-1]: .6e}
      momento  : {momento_arr[0]: .6e}  → {momento_arr[-1]: .6e}
      energía  : {energia_arr[0]: .6e}   → {energia_arr[-1]: .6e}
""").strip()
with open(out_dir / f"{nombre}_explicacion.txt", "w", encoding="utf-8") as f:
    f.write(note)

#print("Listo.")
#print("PNG:", png_path)
#print("PDF:", pdf_path)
#print("Invariantes:", inv_png_path)
#print("Animación:", mp4_path if saved_as=='mp4' else gif_path if saved_as=='gif' else'(no creada)')

/tmp/ipython-input-1383483737.py:36: RuntimeWarning: overflow encountered in multiply
  return - (a * Dx(a) + (delta**2) * Dxxx(a))
/tmp/ipython-input-1383483737.py:28: RuntimeWarning: invalid value encountered in subtract
  return (np.roll(a,-1) - np.roll(a,1))/(2*dx)
/tmp/ipython-input-1383483737.py:30: RuntimeWarning: invalid value encountered in subtract
  return (np.roll(a,-1) - 2*a + np.roll(a,1)) / (dx*dx)
/tmp/ipython-input-1383483737.py:30: RuntimeWarning: invalid value encountered in add
  return (np.roll(a,-1) - 2*a + np.roll(a,1)) / (dx*dx)
/tmp/ipython-input-1383483737.py:36: RuntimeWarning: invalid value encountered in add
  return - (a * Dx(a) + (delta**2) * Dxxx(a))
/tmp/ipython-input-1383483737.py:44: RuntimeWarning: overflow encountered in multiply
  momento = trapz_periodic(a*a)
/tmp/ipython-input-1383483737.py:45: RuntimeWarning: overflow encountered in power
  energia = trapz_periodic((1/3)*a**3 - (delta*phix)*2)
/tmp/ipython-input-1383483737.py:45: RuntimeWarning: